# How do I access wind and temp from GEE?


In [1]:
print("Hello!")

Hello!


In [3]:
import ee
import pandas as pd
# import geemap
print("Authenticating...")
ee.Authenticate()
print("Done. Initializing...")
ee.Initialize(project="climateconsciousimli")
print("Done.")
print("--"*20)
print()

Authenticating...



Successfully saved authorization token.
Done. Initializing...
Done.
----------------------------------------



In [4]:
# Define bounding box (lon_min, lat_min, lon_max, lat_max)
lat_min, lat_max = 28.0, 29.39
lat_avg = (lat_max + lat_min) / 2

lon_min, lon_max = 76.3, 79.0
lon_avg = (lon_max + lon_min) / 2

stepp = 0.25


roi = ee.Geometry.Rectangle([lon_min, lat_min, lon_max, lat_max])
pixel = ee.Geometry.Point([77.0, 29.0])
boxx = ee.Geometry.Rectangle([lon_avg-stepp, lat_avg-stepp, lon_avg+stepp, lat_avg+stepp])
sampling_scale = 5000
start_date = '2024-02-03'
end_date = '2024-02-04'

In [5]:
dataset = (ee.ImageCollection('ECMWF/ERA5/HOURLY')
           .filterBounds(boxx)
           .filterDate(start_date, end_date)
           .select(
                'temperature_2m',
                'u_component_of_wind_10m',
                'v_component_of_wind_10m'
           )
           )

In [6]:
image = dataset.first()

In [7]:
features = image.getInfo()

In [8]:
features

{'type': 'Image',
 'bands': [{'id': 'temperature_2m',
   'data_type': {'type': 'PixelType', 'precision': 'float'},
   'dimensions': [1441, 721],
   'crs': 'EPSG:4326',
   'crs_transform': [0.25, 0, -180.125, 0, -0.25, 90.125]},
  {'id': 'u_component_of_wind_10m',
   'data_type': {'type': 'PixelType', 'precision': 'float'},
   'dimensions': [1441, 721],
   'crs': 'EPSG:4326',
   'crs_transform': [0.25, 0, -180.125, 0, -0.25, 90.125]},
  {'id': 'v_component_of_wind_10m',
   'data_type': {'type': 'PixelType', 'precision': 'float'},
   'dimensions': [1441, 721],
   'crs': 'EPSG:4326',
   'crs_transform': [0.25, 0, -180.125, 0, -0.25, 90.125]}],
 'version': 1755293870123602,
 'id': 'ECMWF/ERA5/HOURLY/20240203T00',
 'properties': {'system:time_start': 1706918400000,
  'hour': 0,
  'system:footprint': {'type': 'LinearRing',
   'coordinates': [[-180, -90],
    [180, -90],
    [180, 90],
    [-180, 90],
    [-180, -90]]},
  'system:time_end': 1706922000000,
  'system:asset_size': 614526828,
  '

In [9]:
img_samples = image.sample(
    region = boxx,
    scale = sampling_scale,
    geometries=True,
    dropNulls=False
)
img_sample_dict = img_samples.getInfo()
img_sample_dict

{'type': 'FeatureCollection',
 'columns': {'temperature_2m': 'Float',
  'u_component_of_wind_10m': 'Float',
  'v_component_of_wind_10m': 'Float'},
 'properties': {'band_order': ['temperature_2m',
   'u_component_of_wind_10m',
   'v_component_of_wind_10m']},
 'features': [{'type': 'Feature',
   'geometry': {'geodesic': False,
    'type': 'Point',
    'coordinates': [77.4444498391698, 28.927271269357597]},
   'id': '0',
   'properties': {'temperature_2m': 283.29461669921875,
    'u_component_of_wind_10m': 2.089021682739258,
    'v_component_of_wind_10m': -1.7515219449996948}},
  {'type': 'Feature',
   'geometry': {'geodesic': False,
    'type': 'Point',
    'coordinates': [77.48936560337575, 28.927271269357597]},
   'id': '1',
   'properties': {'temperature_2m': 283.29461669921875,
    'u_component_of_wind_10m': 2.089021682739258,
    'v_component_of_wind_10m': -1.7515219449996948}},
  {'type': 'Feature',
   'geometry': {'geodesic': False,
    'type': 'Point',
    'coordinates': [77.5342

In [10]:
rows = []
for f in img_sample_dict['features'] : 
    props = f['properties']
    coords = f['geometry']['coordinates']
    props['lat'] = coords[0]
    props['lon'] = coords[1]
    rows.append(props)

img_df = pd.DataFrame(rows)
img_df

,temperature_2m,u_component_of_wind_10m,v_component_of_wind_10m,lat,lon
0,283.294617,2.089022,-1.751522,77.444450,28.927271
1,283.294617,2.089022,-1.751522,77.489366,28.927271
2,283.294617,2.089022,-1.751522,77.534281,28.927271
3,283.294617,2.089022,-1.751522,77.579197,28.927271
4,283.294617,2.089022,-1.751522,77.624113,28.927271
...,...,...,...,...,...
116,283.168091,2.039535,-2.136187,77.713944,28.478114
117,283.168091,2.039535,-2.136187,77.758860,28.478114
118,283.168091,2.039535,-2.136187,77.803776,28.478114
119,283.168091,2.039535,-2.136187,77.848692,28.478114
